**Smart Home Energy Usage Tracker**

Week 4

In [0]:
logs    = spark.read.option("header", True).option("inferSchema", True).csv(f"{BASE}/energy_logs_clean.csv")
devices = spark.read.option("header", True).option("inferSchema", True).csv(f"{BASE}/devices.csv")
rooms   = spark.read.option("header", True).option("inferSchema", True).csv(f"{BASE}/rooms.csv")

print("log rows:", logs.count())
logs.printSchema()

log rows: 9853
root
 |-- log_id: integer (nullable = true)
 |-- device_id: string (nullable = true)
 |-- room_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- power_watts: double (nullable = true)



In [0]:
   %sql
   SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
sasi_008,default


In [0]:
%python
from pyspark.sql import functions as F

CATALOG = "sasi_008"
SCHEMA  = "default"
BASE    = f"/Volumes/{CATALOG}/{SCHEMA}/training_files"

display(dbutils.fs.ls(BASE))

path,name,size,modificationTime
dbfs:/Volumes/sasi_008/default/training_files/README.txt,README.txt,2229,1794376593000
dbfs:/Volumes/sasi_008/default/training_files/attraction_maintenance.csv,attraction_maintenance.csv,1324,1794376593000
dbfs:/Volumes/sasi_008/default/training_files/attractions.json,attractions.json,2562,1794376593000
dbfs:/Volumes/sasi_008/default/training_files/checkpoints/,checkpoints/,0,1794380584000
dbfs:/Volumes/sasi_008/default/training_files/databricks_dataset_setup.py,databricks_dataset_setup.py,3260,1794376593000
dbfs:/Volumes/sasi_008/default/training_files/devices.csv,devices.csv,799,1794480879000
dbfs:/Volumes/sasi_008/default/training_files/energy_logs_clean.csv,energy_logs_clean.csv,433898,1794480863000
dbfs:/Volumes/sasi_008/default/training_files/expenses_clean.csv,expenses_clean.csv,37812,1794479250000
dbfs:/Volumes/sasi_008/default/training_files/output/,output/,0,1794480050000
dbfs:/Volumes/sasi_008/default/training_files/park_entries.csv,park_entries.csv,8592,1794376593000


In [0]:
logs    = spark.read.option("header", True).option("inferSchema", True).csv(f"{BASE}/energy_logs_clean.csv")
devices = spark.read.option("header", True).option("inferSchema", True).csv(f"{BASE}/devices.csv")
rooms   = spark.read.option("header", True).option("inferSchema", True).csv(f"{BASE}/rooms.csv")

print("log rows:", logs.count())
logs.printSchema()

log rows: 9853
root
 |-- log_id: integer (nullable = true)
 |-- device_id: string (nullable = true)
 |-- room_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- power_watts: double (nullable = true)



In [0]:
enriched = (logs
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("week_start", F.date_trunc("week", "timestamp").cast("date"))
    .join(devices.select("device_id", "device_name", "device_type"), "device_id")
    .join(rooms.select("room_id", "room_name"), "room_id"))

display(enriched.limit(10))

room_id,device_id,log_id,timestamp,energy_kwh,power_watts,date,week_start,device_name,device_type,room_name
R01,D01,1,2026-09-01T00:00:00.000Z,1.0275,1027.5,2026-09-01,2026-08-31,Living Room AC,air_conditioner,Living Room
R01,D02,2,2026-09-01T00:00:00.000Z,0.0024,2.4,2026-09-01,2026-08-31,Living Room TV,television,Living Room
R01,D03,3,2026-09-01T00:00:00.000Z,0.0014,1.4,2026-09-01,2026-08-31,Living Room Fan,fan,Living Room
R02,D04,4,2026-09-01T00:00:00.000Z,0.9964,996.4,2026-09-01,2026-08-31,Bedroom 1 AC,air_conditioner,Bedroom 1
R02,D05,5,2026-09-01T00:00:00.000Z,4.0E-4,0.4,2026-09-01,2026-08-31,Bedroom 1 Light,light,Bedroom 1
R03,D06,6,2026-09-01T00:00:00.000Z,0.0713,71.3,2026-09-01,2026-08-31,Bedroom 2 Fan,fan,Bedroom 2
R03,D07,7,2026-09-01T00:00:00.000Z,3.0E-4,0.3,2026-09-01,2026-08-31,Bedroom 2 Light,light,Bedroom 2
R04,D08,8,2026-09-01T00:00:00.000Z,0.071,71.0,2026-09-01,2026-08-31,Refrigerator,refrigerator,Kitchen
R04,D09,9,2026-09-01T00:00:00.000Z,0.02,20.0,2026-09-01,2026-08-31,Microwave,microwave,Kitchen
R04,D10,10,2026-09-01T00:00:00.000Z,5.0E-4,0.5,2026-09-01,2026-08-31,Kitchen Light,light,Kitchen


In [0]:
daily = (enriched.groupBy("date", "device_id", "device_name", "room_name")
         .agg(F.round(F.sum("energy_kwh"), 3).alias("daily_kwh"),
              F.round(F.max("power_watts"), 1).alias("peak_watts"))
         .orderBy("date", F.desc("daily_kwh")))

weekly = (enriched.groupBy("week_start", "room_name")
          .agg(F.round(F.sum("energy_kwh"), 3).alias("weekly_kwh"))
          .orderBy("week_start", F.desc("weekly_kwh")))

display(daily.limit(20))
display(weekly)

date,device_id,device_name,room_name,daily_kwh,peak_watts
2026-09-01,D01,Living Room AC,Living Room,9.526,1306.1
2026-09-01,D04,Bedroom 1 AC,Bedroom 1,7.803,1156.6
2026-09-01,D11,Water Heater,Bathroom,2.765,1844.7
2026-09-01,D08,Refrigerator,Kitchen,1.4,74.9
2026-09-01,D09,Microwave,Kitchen,1.391,931.0
2026-09-01,D02,Living Room TV,Living Room,0.62,128.2
2026-09-01,D06,Bedroom 2 Fan,Bedroom 2,0.619,72.3
2026-09-01,D03,Living Room Fan,Living Room,0.552,71.4
2026-09-01,D12,Office Laptop Charger,Home Office,0.462,71.0
2026-09-01,D13,Office Monitor,Home Office,0.278,43.4


week_start,room_name,weekly_kwh
2026-08-31,Living Room,62.217
2026-08-31,Bedroom 1,41.911
2026-08-31,Bathroom,23.44
2026-08-31,Kitchen,15.715
2026-08-31,Home Office,4.974
2026-08-31,Bedroom 2,3.757
2026-09-07,Living Room,72.067
2026-09-07,Bedroom 1,52.787
2026-09-07,Kitchen,19.353
2026-09-07,Bathroom,18.605


In [0]:
daily.write.format("delta").mode("overwrite").saveAsTable("sasi_008.default.daily_energy_summary")
weekly.write.format("delta").mode("overwrite").saveAsTable("sasi_008.default.weekly_energy_summary")
print("Delta tables saved")

Delta tables saved


In [0]:
daily.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{BASE}/output/daily_summary_csv")
weekly.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{BASE}/output/weekly_summary_csv")
display(dbutils.fs.ls(f"{BASE}/output/daily_summary_csv"))

path,name,size,modificationTime
dbfs:/Volumes/sasi_008/default/training_files/output/daily_summary_csv/_SUCCESS,_SUCCESS,0,1794481274000
dbfs:/Volumes/sasi_008/default/training_files/output/daily_summary_csv/_committed_3440522669744358508,_committed_3440522669744358508,113,1794481274000
dbfs:/Volumes/sasi_008/default/training_files/output/daily_summary_csv/_started_3440522669744358508,_started_3440522669744358508,0,1794481274000
dbfs:/Volumes/sasi_008/default/training_files/output/daily_summary_csv/part-00000-tid-3440522669744358508-e302abfc-54a9-4927-a149-5373328a3208-221-1-c000.csv,part-00000-tid-3440522669744358508-e302abfc-54a9-4927-a149-5373328a3208-221-1-c000.csv,21625,1794481274000


In [0]:
%sql
SELECT device_name, room_name, ROUND(SUM(daily_kwh), 2) AS total_kwh, COUNT(*) AS days
FROM sasi_008.default.daily_energy_summary
GROUP BY device_name, room_name
ORDER BY total_kwh DESC
LIMIT 5;

device_name,room_name,total_kwh,days
Living Room AC,Living Room,271.24,30
Bedroom 1 AC,Bedroom 1,232.68,30
Water Heater,Bathroom,102.45,30
Refrigerator,Kitchen,39.5,30
Microwave,Kitchen,28.15,30
